# Thinking Out Loud: within-session nested CV

This standalone Kaggle notebook tests whether training a separate model for each recording session changes generalization compared with pooling sessions of the same subject. It reads the derivative EEG directly and creates its own splits. No previous notebook, saved splits, or feature exports are required.

**Default experiment:** subjects 1–2; all sessions; 128 EEG channels; the same 0.5–3.0 s exclusive cue-relative window and five log-bandpower features per channel as your reviewed baseline. Both ANOVA and MRMR choose k, C, and either linear SVM or logistic regression using inner CV.

Each within-session model sees training trials from exactly one session. A new pooled comparator trains on the union of the session training folds and predicts **the same held-out trials**. Outer folds and inner folds are formed within each session and then combined for the comparator. Neither model sees any outer-test trial during fitting.

This comparator deliberately uses matched splits, so it does **not** reproduce the previous pooled notebook's class-only split plan. Within-session training also changes training-set size and allows a different model for each session; a score difference is evidence about these procedures, not proof that session mixing alone caused the earlier result. This is an exploratory comparison prompted by previously observed test scores. Neither mode is LOSO or evaluation on a new session.

Run cells in order. Start with the defaults, inspect the output, then change `SUBJECTS` to `list(range(1, 11))` to extend the same experiment. Keep the data files and output folder available to resume. A fresh Kaggle session must have the saved output folder restored if you want to resume previous work.


## 1. Dependencies
Only missing EEG/MRMR packages are installed; this cell does not upgrade installed packages. Kaggle Internet access is required if a package is absent. Your previous Kaggle environment's NumPy 2.x supports the unchanged trapezoid band integration. CPU execution is sufficient; a GPU is not used.


In [ ]:
import importlib.util
import subprocess
import sys

for module, package in [('mne', 'mne'), ('mrmr', 'mrmr-selection')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', package])

import numpy as np
if not hasattr(np, 'trapezoid'):
    raise RuntimeError('This notebook needs NumPy >= 2.0, matching your reviewed Kaggle environment.')


## 2. Configuration

- `SUBJECTS`: `[1, 2]` initially; later `list(range(1, 11))`.
- `SESSIONS=None`: use all available sessions, evaluated separately. For one session only, use `[1]` and disable the pooled comparator.
- `RUN_POOLED_COMPARATOR=True`: recommended for the question you are testing; held-out trials match exactly between modes.
- `SELECTORS=['anova', 'mrmr']`: matches your latest comparison. `['anova']` is a faster preliminary check, but record that restriction before looking at results.
- `CLASSIFIERS=['linear_svm', 'logistic']`: matches your previous inner-selected classifier procedure. Use `['linear_svm']` only if you want a separate SVM-only experiment.
- Leave the window, C/k grid, and fold counts fixed during this comparison. MRMR may be slow; candidate-level progress and completed-fold checkpoints are enabled.

Configuration/data/code changes create distinct run identities. Extending the subject list can reuse completed subjects under the same scientific configuration. This notebook uses its own output root and does not modify your old results.


In [ ]:
from pathlib import Path
from IPython.display import display, FileLink

DATA_ROOT = Path('/kaggle/input/datasets/truthisneverlinear/inner-speech-recognition/inner-speech-recognition/derivatives')
OUTPUT_ROOT = Path('/kaggle/working/within_session_diagnostics_results')

SUBJECTS = [1, 2]
SESSIONS = None
RANDOM_SEED = 42
CHANNEL_COUNT = 128
ACTION_WINDOW_MNE = (0.5, 3.0)
ACTION_ENDPOINT = 'exclusive'
OUTER_N_SPLITS = 5
INNER_N_SPLITS = 5

C_VALUES = [0.001, 0.01, 0.1, 1.0]
K_VALUES = [20, 40, 80, 120]
SELECTORS = ['anova', 'mrmr']
CLASSIFIERS = ['linear_svm', 'logistic']
RUN_POOLED_COMPARATOR = True
N_JOBS = 1
SEARCH_VERBOSE = 2
RESUME = True

assert CHANNEL_COUNT == 128
assert ACTION_ENDPOINT == 'exclusive'
assert ACTION_WINDOW_MNE == (0.5, 3.0)

CONFIG = dict(data_root=str(DATA_ROOT), output_root=str(OUTPUT_ROOT),
    subjects=SUBJECTS, sessions=SESSIONS, seed=RANDOM_SEED,
    action_window_mne=list(ACTION_WINDOW_MNE), endpoint=ACTION_ENDPOINT,
    outer_n_splits=OUTER_N_SPLITS, inner_n_splits=INNER_N_SPLITS,
    c_values=C_VALUES, k_values=K_VALUES, selectors=SELECTORS,
    classifiers=CLASSIFIERS, run_pooled_comparator=RUN_POOLED_COMPARATOR,
    n_jobs=N_JOBS, search_verbose=SEARCH_VERBOSE, resume=RESUME)
print(CONFIG)


## 3. Self-contained pipeline
The code below is embedded in this notebook. It creates an importable helper module in Kaggle working storage so scikit-learn can cache training-fold transformations reliably. No external project file is needed. The executed source is included in the review ZIP, including both selector classes.

Feature extraction is per trial and uses no labels. Variance filtering, scaling, ANOVA/MRMR, and classification are inside the inner-CV pipeline. Splits are checked for disjointness, complete coverage, and session boundaries before fitting. The original full-channel feature indices and channel/band names selected at each outer refit are saved. Feature selection is not channel reduction.


In [ ]:
CORE_SOURCE = r'''
"""Independent within-session EEG evaluation, with a matched pooled comparator."""
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
from zipfile import ZipFile, ZIP_DEFLATED
from importlib.metadata import version, PackageNotFoundError
import hashlib
import json
import warnings
import traceback
import platform
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.signal import welch
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV, ParameterGrid
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from threadpoolctl import threadpool_limits

CORE_VERSION = 'within-session-1.0'
BANDS = ((1.,4.), (4.,8.), (8.,13.), (13.,30.), (30.,45.))

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str,
                                    separators=(',', ':')).encode()).hexdigest()

def array_digest(value):
    value = np.ascontiguousarray(value)
    h = hashlib.sha256(str((value.shape, value.dtype.str)).encode())
    h.update(memoryview(value).cast('B'))
    return h.hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, default=str), encoding='utf-8')
    tmp.replace(path)

def environment():
    result = {'python': platform.python_version(), 'numpy': np.__version__,
              'pandas': pd.__version__, 'scipy': scipy.__version__,
              'sklearn': sklearn.__version__}
    for name in ('mne', 'mrmr-selection', 'joblib'):
        try:
            result[name] = version(name)
        except PackageNotFoundError:
            result[name] = 'not installed'
    return result

def bandpower_features(X, sfreq):
    n = min(int(round(sfreq)), X.shape[-1])
    if sfreq / 2 < 45 or n < 16:
        raise ValueError('Insufficient sampling rate or epoch length for declared bands.')
    freqs, density = welch(X, fs=sfreq, window='hann', nperseg=n,
                          noverlap=n//2, detrend='constant', scaling='density', axis=-1)
    values = []
    for low, high in BANDS:
        mask = (freqs >= low) & (freqs <= high)
        if mask.sum() < 2:
            raise ValueError('Insufficient frequency bins.')
        power = np.trapezoid(density[...,mask], freqs[mask], axis=-1)
        values.append(np.log10(np.maximum(power, 1e-24)))
    return np.stack(values, axis=-1).reshape(len(X), -1).astype(np.float64)

def prepare_subject(subject, config):
    import mne
    root = Path(config['data_root'])
    files = sorted(root.rglob(f'sub-{subject:02d}_ses-*_eeg-epo.fif'))
    if not files:
        raise FileNotFoundError(f'No EEG epoch files for subject {subject} below {root}')
    matrices, records, source = [], [], []
    reference = None
    seen = set()
    for path in files:
        session = int(path.name.split('_')[1].split('-')[1])
        if config['sessions'] is not None and session not in config['sessions']:
            continue
        if session in seen:
            raise ValueError(f'Duplicate session {session}: select a single dataset version.')
        seen.add(session)
        event_path = path.with_name(path.name.replace('_eeg-epo.fif', '_events.dat'))
        # Public dataset event files are pickles; use only your trusted dataset copy.
        raw_events = np.asarray(pd.read_pickle(event_path))
        epochs = mne.read_epochs(path, preload=False, proj=False, verbose='ERROR')
        if raw_events.shape != (len(epochs), 4):
            raise ValueError(f'Event dimensions do not match epochs: {path.name}')
        if not np.isfinite(raw_events).all() or not np.equal(raw_events, np.floor(raw_events)).all():
            raise ValueError('Event values must be finite integers.')
        events = raw_events.astype(np.int64)
        if not np.array_equal(events[:,0], epochs.events[:,0]) or not np.array_equal(events[:,1]+31, epochs.events[:,2]):
            raise ValueError('Source event/epoch alignment failed.')
        if not np.isin(events[:,1], [0,1,2,3]).all():
            raise ValueError('Unexpected direction label.')
        original_rows = np.asarray(epochs.selection)
        selected = np.flatnonzero(events[:,2] == 1)
        epochs = epochs[selected].load_data()
        epochs.pick(mne.pick_types(epochs.info, eeg=True, exclude=[]))
        if len(epochs.ch_names) != 128:
            raise ValueError('Exactly 128 EEG channels are required.')
        start, end = config['action_window_mne']
        mask = (epochs.times >= start) & (epochs.times < end)
        times, sfreq, names = epochs.times[mask], float(epochs.info['sfreq']), list(epochs.ch_names)
        if len(times) == 0:
            raise ValueError('Empty action window.')
        if reference is not None and (names != reference[0] or sfreq != reference[1] or not np.array_equal(times,reference[2])):
            raise ValueError('Channels, sampling, or epoch times differ across sessions.')
        reference = names, sfreq, times
        X = epochs.get_data(copy=False)[:,:,mask].astype(np.float32)
        if not np.isfinite(X).all():
            raise ValueError('Nonfinite EEG values.')
        matrices.append(bandpower_features(X, sfreq))
        for row, event in zip(original_rows[selected], events[selected]):
            records.append(dict(subject=int(subject), session=session, trial_index=int(row),
                cue_sample_raw=int(event[0]), class_id=int(event[1]),
                trial_id=f'sub-{subject:02d}_ses-{session:02d}_row-{row:04d}'))
        source.append(dict(eeg=str(path), events=str(event_path), inner_trials=len(X),
            X_sha256=array_digest(X), sfreq=sfreq, n_times=len(times),
            first_time=float(times[0]), last_time=float(times[-1]),
            RMS_uV_median=float(np.median(np.sqrt(np.mean(X.astype(float)**2, axis=(1,2)))*1e6))))
        print(f'Subject {subject:02d}, session {session:02d}: {len(X)} trials, {len(names)} channels, {len(times)} samples', flush=True)
    if not matrices:
        raise ValueError('No sessions matched SESSIONS.')
    if config['sessions'] is not None and seen != set(config['sessions']):
        raise ValueError(f'Requested sessions missing: {set(config["sessions"])-seen}')
    metadata = pd.DataFrame(records)
    F = np.concatenate(matrices)
    if metadata.trial_id.duplicated().any() or not np.isfinite(F).all():
        raise ValueError('Duplicate IDs or nonfinite features.')
    return dict(subject=int(subject), F=F, y=metadata.class_id.to_numpy(dtype=np.int64),
                trial_metadata=metadata, ch_names=names, provenance=source)

def stratified_parts(indices, y, n_splits, seed):
    indices = np.asarray(indices, dtype=int)
    counts = np.bincount(y[indices], minlength=4)
    if (counts < n_splits).any():
        raise ValueError(f'Class counts {counts.tolist()} cannot support {n_splits} folds. No automatic fold-count change.')
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return [(indices[a], indices[b]) for a,b in cv.split(np.zeros(len(indices)),y[indices])]

def make_plan(data, config):
    y, metadata = data['y'], data['trial_metadata']
    session_values = sorted(metadata.session.unique().tolist())
    per_session = {s: stratified_parts(np.flatnonzero(metadata.session.to_numpy()==s),y,
                    config['outer_n_splits'], config['seed']) for s in session_values}
    tasks = []
    for fold in range(config['outer_n_splits']):
        inner_by_session = {}
        for s in session_values:
            train, test = per_session[s][fold]
            inner = stratified_parts(train,y,config['inner_n_splits'],config['seed']+fold+1)
            inner_by_session[s] = inner
            tasks.append(dict(mode='within_session', session=int(s), outer_fold=fold+1,
                train=train.tolist(), test=test.tolist(),
                inner=[dict(train=a.tolist(), validation=b.tolist()) for a,b in inner]))
        if config['run_pooled_comparator']:
            tasks.append(dict(mode='pooled_sessions_matched', session=0, outer_fold=fold+1,
                train=np.concatenate([per_session[s][fold][0] for s in session_values]).tolist(),
                test=np.concatenate([per_session[s][fold][1] for s in session_values]).tolist(),
                inner=[dict(train=np.concatenate([inner_by_session[s][j][0] for s in session_values]).tolist(),
                            validation=np.concatenate([inner_by_session[s][j][1] for s in session_values]).tolist())
                       for j in range(config['inner_n_splits'])]))
    plan = dict(subject=data['subject'], seed=config['seed'], trial_ids=metadata.trial_id.tolist(),
                labels=y.tolist(), outer_n_splits=config['outer_n_splits'],
                inner_n_splits=config['inner_n_splits'], tasks=tasks)
    validate_plan(data,plan)
    return plan

def validate_plan(data, plan):
    metadata, y = data['trial_metadata'], data['y']
    if plan['trial_ids'] != metadata.trial_id.tolist() or plan['labels'] != y.tolist():
        raise ValueError('Plan alignment failure.')
    for mode in {t['mode'] for t in plan['tasks']}:
        coverage = np.zeros(len(y),dtype=int)
        for task in [t for t in plan['tasks'] if t['mode']==mode]:
            a,b = set(task['train']),set(task['test'])
            universe = (set(np.flatnonzero(metadata.session.to_numpy()==task['session']))
                        if mode=='within_session' else set(range(len(y))))
            if a & b or a | b != universe or len(a)!=len(task['train']) or len(b)!=len(task['test']):
                raise ValueError('Invalid outer partition.')
            coverage[task['test']] += 1
            inner_coverage = dict.fromkeys(a,0)
            for inner in task['inner']:
                c,d = set(inner['train']),set(inner['validation'])
                if c & d or c | d != a or (c|d)&b or len(c)!=len(inner['train']) or len(d)!=len(inner['validation']):
                    raise ValueError('Invalid inner partition or outer-test leakage.')
                if set(y[list(c)]) != {0,1,2,3} or set(y[list(d)]) != {0,1,2,3}:
                    raise ValueError('Inner fold missing a class.')
                for i in d:
                    inner_coverage[i] += 1
            if set(inner_coverage.values()) != {1}:
                raise ValueError('Inner validation coverage is not exactly once.')
        if not np.all(coverage==1):
            raise ValueError('Outer test coverage is not exactly once per mode.')
    for pooled in [t for t in plan['tasks'] if t['mode']=='pooled_sessions_matched']:
        within = [t for t in plan['tasks'] if t['mode']=='within_session' and t['outer_fold']==pooled['outer_fold']]
        if set(pooled['test']) != {i for t in within for i in t['test']}:
            raise ValueError('Comparison test trials differ.')

class BoundedANOVA(BaseEstimator, TransformerMixin):
    def __init__(self, k=40):
        self.k = k
    def fit(self, X, y):
        self.selector_ = SelectKBest(f_classif,k=min(int(self.k),X.shape[1])).fit(X,y)
        return self
    def transform(self, X):
        return self.selector_.transform(X)
    def get_support(self):
        return self.selector_.get_support()

class FoldMRMR(BaseEstimator, TransformerMixin):
    def __init__(self, k=40):
        self.k = k
    def fit(self, X, y):
        from mrmr import mrmr_classif
        names = [f'f{i}' for i in range(X.shape[1])]
        chosen = mrmr_classif(X=pd.DataFrame(X,columns=names), y=pd.Series(y),
            K=min(int(self.k),X.shape[1]), relevance='f', redundancy='c',
            n_jobs=1, show_progress=False)
        self.selected_ = np.asarray([names.index(n) for n in chosen],dtype=int)
        self.n_features_in_ = X.shape[1]
        if len(self.selected_) == 0:
            raise ValueError('MRMR returned no features.')
        return self
    def transform(self, X):
        return X[:,self.selected_]
    def get_support(self):
        result = np.zeros(self.n_features_in_,dtype=bool)
        result[self.selected_] = True
        return result

def make_grid(config):
    result = []
    for name in config['classifiers']:
        if name=='linear_svm':
            model = LinearSVC(dual='auto',max_iter=20000,random_state=config['seed'])
        elif name=='logistic':
            model = LogisticRegression(solver='lbfgs',max_iter=5000,random_state=config['seed'])
        else:
            raise ValueError(f'Unknown classifier {name}')
        result.append({'model':[model], 'model__C':config['c_values'], 'selector__k':config['k_values']})
    return result

def simple_refit_index(results):
    scores = np.asarray(results['mean_test_score'])
    if not np.isfinite(scores).all():
        raise ValueError('Nonfinite inner-CV score.')
    tied = np.flatnonzero(np.isclose(scores,scores.max(),rtol=0,atol=1e-12))
    return int(min(tied,key=lambda i:(int(results['params'][i]['selector__k']),
                    float(results['params'][i]['model__C']),i)))

def evaluate_task(data, plan, task, selector, config, run_root, run_id):
    task_key = f"{task['mode']}_ses-{task['session']:02d}_{selector}_fold-{task['outer_fold']}"
    path = run_root/'checkpoints'/f'{task_key}.json'
    train,test = np.asarray(task['train']),np.asarray(task['test'])
    F,y,metadata = data['F'],data['y'],data['trial_metadata']
    expected_ids = metadata.trial_id.iloc[test].tolist()
    if config['resume'] and path.exists():
        saved = json.loads(path.read_text())
        pred = saved['predictions']
        if (saved['run_id'] != run_id or saved['task_digest'] != digest(task)
            or [p['trial_id'] for p in pred] != expected_ids
            or [p['y_true'] for p in pred] != y[test].tolist()
            or any(p['y_pred'] not in [0,1,2,3] for p in pred)
            or len(pred)!=len(test)):
            raise ValueError(f'Invalid checkpoint: {path}')
        print(f'Resumed {task_key}',flush=True)
        return saved
    lookup = {int(index):local for local,index in enumerate(train)}
    inner = [(np.array([lookup[i] for i in v['train']]),
              np.array([lookup[i] for i in v['validation']])) for v in task['inner']]
    transform = BoundedANOVA() if selector=='anova' else FoldMRMR()
    pipeline = Pipeline([('variance',VarianceThreshold(0.0)),('scaler',StandardScaler()),
                         ('selector',transform),('model',LinearSVC())],
                        memory=str(run_root/'cache'/digest(task_key)[:12]))
    grid = make_grid(config)
    n_candidates = len(ParameterGrid(grid))
    print(f"\nSubject {data['subject']:02d} | {task_key} | train={len(train)} test={len(test)} | "
          f"{n_candidates} candidates x {len(inner)} inner folds",flush=True)
    search = GridSearchCV(pipeline,grid,cv=inner,scoring='balanced_accuracy',
        refit=simple_refit_index,return_train_score=True,error_score='raise',
        n_jobs=config['n_jobs'],verbose=config['search_verbose'])
    started = perf_counter()
    with warnings.catch_warnings(record=True) as caught, threadpool_limits(limits=1):
        warnings.simplefilter('always')
        search.fit(F[train],y[train])
        model = search.best_estimator_
        train_pred,test_pred = model.predict(F[train]),model.predict(F[test])
    elapsed = perf_counter()-started
    selected = np.flatnonzero(model.named_steps['variance'].get_support())[
        model.named_steps['selector'].get_support()]
    classifier = 'linear_svm' if isinstance(model.named_steps['model'],LinearSVC) else 'logistic'
    common = dict(run_id=run_id,subject=data['subject'],mode=task['mode'],selector=selector,
                  outer_fold=task['outer_fold'])
    predictions = [dict(**common, session=int(metadata.session.iloc[i]),
        trial_id=str(metadata.trial_id.iloc[i]),y_true=int(y[i]),y_pred=int(p)) for i,p in zip(test,test_pred)]
    rows = []
    for session in sorted(metadata.session.iloc[test].unique()):
        mask = metadata.session.iloc[test].to_numpy()==session
        train_mask = metadata.session.iloc[train].to_numpy()==session
        rows.append(dict(**common,session=int(session),classifier=classifier,
            requested_k=int(search.best_params_['selector__k']),num_features=int(len(selected)),
            best_C=float(search.best_params_['model__C']),n_train=len(train),n_test=int(mask.sum()),
            n_train_target_session=int(train_mask.sum()),
            train_balanced_accuracy=float(balanced_accuracy_score(y[train],train_pred)),
            train_target_session_BA=float(balanced_accuracy_score(y[train][train_mask],train_pred[train_mask])),
            test_balanced_accuracy=float(balanced_accuracy_score(y[test][mask],test_pred[mask])),
            inner_best_balanced_accuracy=float(search.cv_results_['mean_test_score'][search.best_index_]),
            fit_seconds=elapsed))
    search_dir = run_root/'inner_search'
    search_dir.mkdir(exist_ok=True)
    pd.DataFrame(search.cv_results_).to_csv(search_dir/f'{task_key}.csv',index=False)
    saved = dict(run_id=run_id,task_digest=digest(task),rows=rows,predictions=predictions,
                 selected_feature_indices=selected.tolist(),
                 selected_features=[dict(channel=data['ch_names'][i//5],band=list(BANDS[i%5])) for i in selected],
                 warnings=sorted({str(w.message) for w in caught}),fit_seconds=elapsed)
    save_json(path,saved)  # Written last: an interrupted fit never becomes a completed checkpoint.
    print(f"Finished in {elapsed/60:.2f} min | " + ' | '.join(
        f"ses-{r['session']:02d}: test BA={r['test_balanced_accuracy']:.3f}" for r in rows),flush=True)
    return saved

def summarize(completed, expected, output_dir):
    rows = pd.DataFrame([r for result in completed for r in result['rows']])
    predictions = pd.DataFrame([p for result in completed for p in result['predictions']])
    rows.to_csv(output_dir/'fold_results.csv',index=False)
    predictions.to_csv(output_dir/'oof_predictions.csv',index=False)
    summary = []
    for item in expected:
        key = {k:item[k] for k in ['run_id','subject','session','mode','selector']}
        if len(predictions):
            p = predictions.loc[np.logical_and.reduce([predictions[k].eq(v) for k,v in key.items()])]
            r = rows.loc[np.logical_and.reduce([rows[k].eq(v) for k,v in key.items()])]
        else:
            p,r = pd.DataFrame(),pd.DataFrame()
        complete = (len(p)==len(item['trial_ids']) and not p.trial_id.duplicated().any()
                    and set(p.trial_id)==set(item['trial_ids']) and len(r)==item['expected_folds']) if len(p) else False
        summary.append(dict(**key,complete=complete,completed_folds=len(r),expected_folds=item['expected_folds'],
            predicted_trials=len(p),expected_trials=len(item['trial_ids']),
            test_BA=balanced_accuracy_score(p.y_true,p.y_pred) if complete else np.nan,
            train_BA_mean=r.train_target_session_BA.mean() if len(r) else np.nan,
            train_trials_mean=r.n_train.mean() if len(r) else np.nan,
            confusion_matrix=json.dumps(confusion_matrix(p.y_true,p.y_pred,labels=[0,1,2,3]).tolist()) if complete else ''))
    summary = pd.DataFrame(summary)
    summary.to_csv(output_dir/'session_summary.csv',index=False)
    comparisons = []
    if len(summary):
        for keys,g in summary.groupby(['run_id','subject','session','selector']):
            complete = g.loc[g.complete].set_index('mode')
            if {'within_session','pooled_sessions_matched'}.issubset(complete.index):
                comparisons.append(dict(zip(['run_id','subject','session','selector'],keys),
                    within_session_BA=float(complete.loc['within_session','test_BA']),
                    pooled_matched_BA=float(complete.loc['pooled_sessions_matched','test_BA']),
                    within_minus_pooled_BA=float(complete.loc['within_session','test_BA']-complete.loc['pooled_sessions_matched','test_BA'])))
    comparison = pd.DataFrame(comparisons)
    comparison.to_csv(output_dir/'paired_comparison.csv',index=False)
    return summary,comparison

def export_review(output_dir):
    output_dir = Path(output_dir)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
    archive = output_dir.parent/f'within_session_review_{stamp}.zip'
    expected_path = output_dir/'expected_results.json'
    active = ({r['run_id'] for r in json.loads(expected_path.read_text())}
              if expected_path.exists() else None)
    with ZipFile(archive,'w',ZIP_DEFLATED) as z:
        for path in sorted(output_dir.rglob('*')):
            relative = path.relative_to(output_dir)
            if relative.parts[0]=='runs' and len(relative.parts)>1 and active is not None and relative.parts[1] not in active:
                continue
            if path.is_file() and 'cache' not in relative.parts and not path.name.endswith('.tmp'):
                z.write(path,relative)
    print(f'Review ZIP: {archive}',flush=True)
    return archive

def run_experiment(config):
    config = dict(config)
    if not config['subjects'] or len(set(config['subjects']))!=len(config['subjects']):
        raise ValueError('SUBJECTS must contain unique subject numbers.')
    if not set(config['subjects']).issubset(range(1,11)):
        raise ValueError('Subject numbers must be 1 through 10.')
    if not config['selectors'] or not set(config['selectors']).issubset({'anova','mrmr'}):
        raise ValueError('SELECTORS must contain anova and/or mrmr.')
    if config['outer_n_splits'] < 2 or config['inner_n_splits'] < 2:
        raise ValueError('CV requires at least two folds.')
    if not config['k_values'] or any(int(k)!=k or k<1 for k in config['k_values']):
        raise ValueError('Feature counts must be positive integers.')
    if not config['c_values'] or any(c<=0 for c in config['c_values']):
        raise ValueError('C values must be positive.')
    if 'mrmr' in config['selectors']:
        from mrmr import mrmr_classif  # Fail before loading/fitting if dependency is missing.
    source = Path(__file__).read_text(encoding='utf-8')
    scientific_config = {k:v for k,v in config.items() if k not in ['resume','n_jobs','search_verbose','output_root','data_root','subjects']}
    identity = dict(core=CORE_VERSION,source_sha256=hashlib.sha256(source.encode()).hexdigest(),
                    scientific_config=scientific_config,environment=environment())
    output_dir = Path(config['output_root'])/f'experiment-{digest(identity)[:16]}'
    output_dir.mkdir(parents=True,exist_ok=True)
    (output_dir/'executed_core.py').write_text(source,encoding='utf-8')
    save_json(output_dir/'configuration.json',config)
    save_json(output_dir/'environment.json',environment())
    completed,expected,errors = [],[],[]
    summary,comparison = pd.DataFrame(),pd.DataFrame()
    try:
        for subject in config['subjects']:
            data = prepare_subject(subject,config)
            if config['run_pooled_comparator'] and data['trial_metadata'].session.nunique()<2:
                raise ValueError('The pooled comparator needs at least two sessions. Set SESSIONS=None or RUN_POOLED_COMPARATOR=False.')
            plan = make_plan(data,config)
            manifest = dict(**identity,subject=int(subject),features_sha256=array_digest(data['F']),
                labels_sha256=array_digest(data['y']),plan_sha256=digest(plan),
                trial_ids=data['trial_metadata'].trial_id.tolist(),channels=data['ch_names'])
            run_id = digest(manifest)[:24]
            run_root = output_dir/'runs'/run_id
            run_root.mkdir(parents=True,exist_ok=True)
            save_json(run_root/'manifest.json',manifest)
            save_json(run_root/'plan.json',plan)
            save_json(run_root/'input_provenance.json',data['provenance'])
            data['trial_metadata'].to_csv(run_root/'trial_metadata.csv',index=False)
            np.savez_compressed(run_root/'features.npz',F=data['F'],y=data['y'],
                                trial_ids=np.asarray(plan['trial_ids']),ch_names=np.asarray(data['ch_names']))
            modes = sorted({t['mode'] for t in plan['tasks']})
            for session,g in data['trial_metadata'].groupby('session'):
                for mode in modes:
                    for selector in config['selectors']:
                        expected.append(dict(run_id=run_id,subject=int(subject),session=int(session),mode=mode,
                            selector=selector,trial_ids=g.trial_id.tolist(),expected_folds=config['outer_n_splits']))
            for selector in config['selectors']:
                for task in plan['tasks']:
                    completed.append(evaluate_task(data,plan,task,selector,config,run_root,run_id))
                    summary,comparison = summarize(completed,expected,output_dir)
    except BaseException as exc:
        errors.append(dict(type=type(exc).__name__,message=str(exc),traceback=traceback.format_exc()))
        raise
    finally:
        summary,comparison = summarize(completed,expected,output_dir)
        save_json(output_dir/'expected_results.json',expected)
        save_json(output_dir/'execution_status.json',dict(errors=errors,subjects_requested=config['subjects'],
            completed_tasks=len(completed),all_requested_completed=bool(len(summary) and summary.complete.all() and not errors),
            note='Incomplete rows have no final test BA. Existing outer-test results motivated this exploratory comparison.'))
        export_review(output_dir)
    return dict(output_dir=output_dir,session_summary=summary,paired_comparison=comparison)
'''

import importlib
module_dir = Path('/kaggle/working/within_session_pipeline_code')
module_dir.mkdir(parents=True, exist_ok=True)
module_path = module_dir / 'within_session_core.py'
module_path.write_text(CORE_SOURCE, encoding='utf-8')
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))
import within_session_core as pipeline_core
pipeline_core = importlib.reload(pipeline_core)
print('Pipeline ready:', pipeline_core.CORE_VERSION)
print(pipeline_core.environment())


## 4. Inspect the trial counts and planned training sizes
This cell loads only the first subject, computes the fixed features, and validates the plan without training models. `subject_data`, `F`, `y`, and `trial_metadata` remain available for your own exploration.

For a session with 40 trials, five-fold outer CV leaves only 32 training trials. Five-fold inner CV divides those 32 again. A run stops with an explanation if any class cannot support the declared folds; it never silently changes the evaluation design.


In [ ]:
subject_data = pipeline_core.prepare_subject(SUBJECTS[0], CONFIG)
F = subject_data['F']
y = subject_data['y']
trial_metadata = subject_data['trial_metadata']
plan = pipeline_core.make_plan(subject_data, CONFIG)

display(trial_metadata.groupby(['subject', 'session', 'class_id']).size().unstack('class_id', fill_value=0))
import pandas as pd
display(pd.DataFrame([
    dict(mode=t['mode'], session=t['session'], outer_fold=t['outer_fold'],
         training_trials=len(t['train']), test_trials=len(t['test']),
         smallest_inner_training_set=min(len(i['train']) for i in t['inner']))
    for t in plan['tasks']
]))
print('F:', F.shape, '| y:', y.shape)
print('Session 0 denotes the pooled comparator, which predicts each session separately.')


## 5. Run the experiment
This is the training cell. It reloads each requested subject, writes the split plan and fingerprints, and evaluates all configured procedures. Each fit uses inner CV to select parameters; the outer test is used only for prediction and scoring.

Each completed outer task is saved atomically. With unchanged settings and `RESUME=True`, rerunning this cell reuses those checkpoints. A normal error or keyboard interruption also exports the completed results and a traceback; a forcibly terminated Kaggle process cannot execute the export handler, but completed checkpoints remain on disk.


In [ ]:
experiment_result = pipeline_core.run_experiment(CONFIG)
session_summary = experiment_result['session_summary']
paired_comparison = experiment_result['paired_comparison']
print('Saved to:', experiment_result['output_dir'])
display(session_summary)
display(paired_comparison)


## 6. Read the comparison
`test_BA` is balanced accuracy from all out-of-fold predictions for one subject/session/procedure; 0.25 is the four-class chance reference. Missing folds produce an incomplete result with no final score. Training scores are resubstitution scores and should not be interpreted as generalization.

`within_minus_pooled_BA > 0` means within-session training scored higher on the same held-out trials. Inspect the pattern across sessions and subjects, not only the best row. The session scores and CV folds are not independent participants, so this notebook does not perform a naive t-test over folds or declare significance. No shuffled-label controls are run here.

If a later method is designed after reviewing these scores, report it as a new exploratory experiment and protect its final evaluation from further test-informed tuning.


In [ ]:
import matplotlib.pyplot as plt

if paired_comparison.empty:
    print('No complete paired comparisons yet (or the comparator is disabled).')
else:
    plot_data = paired_comparison.copy()
    plot_data['label'] = plot_data.apply(
        lambda r: f"S{int(r.subject):02d}/ses-{int(r.session):02d} {r.selector}", axis=1)
    ax = plot_data.set_index('label')[['within_session_BA', 'pooled_matched_BA']].plot.bar(
        figsize=(max(8, len(plot_data) * 0.65), 4), ylim=(0, 1), rot=60)
    ax.axhline(0.25, color='black', linestyle='--', linewidth=1, label='Chance reference')
    ax.set_ylabel('Out-of-fold balanced accuracy')
    ax.set_xlabel('Subject / session / selector')
    ax.legend()
    plt.tight_layout()
    plt.savefig(experiment_result['output_dir'] / 'session_comparison.png', dpi=160)
    plt.show()


## 7. Download and share the review ZIP
The ZIP contains configuration, package versions, executed source, split plans, input fingerprints and features, selected features, inner-search scores, warnings, fold scores, trial-level predictions, session summaries, and the paired comparison. Cache files are excluded.

After an interrupted run, this cell can package the experiment folders even if `experiment_result` was never assigned. Only existing files are packaged; nothing is retrained. Download the ZIP from the displayed link or Kaggle's Output panel and send it in our conversation.

Implementation references: [scikit-learn Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), [avoiding leakage](https://scikit-learn.org/stable/common_pitfalls.html), [mrmr-selection](https://github.com/smazzanti/mrmr).


In [ ]:
if 'experiment_result' in globals():
    export_dirs = [experiment_result['output_dir']]
else:
    export_dirs = sorted(OUTPUT_ROOT.glob('experiment-*'))
if not export_dirs:
    print('No experiment output exists yet. Run the training cell first.')
for directory in export_dirs:
    review_zip = pipeline_core.export_review(directory)
    display(FileLink(str(review_zip)))
